# Phase 3: Socratic QLoRA Fine-Tuning for Qwen2.5-7B-Instruct

This notebook is designed to run on a **Google Colab Free T4 GPU (16GB VRAM)**.

### Step 1: Upload your dataset
1. On the left sidebar, click the **Folder icon** (Files).
2. Drag and drop your `sft_train.jsonl` and `sft_eval.jsonl` files into the files area.
3. Wait for them to finish uploading before running the cells below.

In [ ]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets
!pip install -q flash-attn --no-build-isolation

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import json
import csv
from transformers import TrainerCallback

MODEL_PATH = 'Qwen/Qwen2.5-7B-Instruct'
OUTPUT_DIR = './checkpoints/socratic-sft'

In [ ]:
def load_socratic_dataset(path, tokenizer, max_length=4096):
    examples = []
    with open(path) as f:
        for line in f:
            ex = json.loads(line)
            text = tokenizer.apply_chat_template(
                ex['messages'],
                tokenize=False,
                add_generation_prompt=False
            )
            examples.append({'text': text})
    return Dataset.from_list(examples)

In [ ]:
# ── 4-bit quantization ──────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(model)

In [ ]:
# ── LoRA adapter ────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Load datasets ───────────────────────────────────────────────
train_ds = load_socratic_dataset('sft_train.jsonl', tokenizer)
eval_ds  = load_socratic_dataset('sft_eval.jsonl', tokenizer)

In [ ]:
# ── Training config ─────────────────────────────────────────────
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,                  # Changed to 2 epochs since baseline was 100%
    per_device_train_batch_size=1,       # Lowered to 1 to fit 16GB Colab T4
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,      # Increased to 16 to keep effective batch size at 16
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    bf16=False,                          # T4 GPU doesn't natively support bf16 well, but we can try or use fp16
    fp16=True,                           # Changed to fp16 for T4 compatibility
    gradient_checkpointing=True,
    logging_steps=10,
    save_steps=150,                      # Save more frequently on Colab
    eval_strategy='steps',
    eval_steps=150,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    run_name='socratic-sft-qwen25-7b',
)

class CSVLoggerCallback(TrainerCallback):
    def __init__(self, path='training_log.csv'):
        self.path = path
        with open(self.path, 'w', newline='') as f:
            csv.writer(f).writerow(['step','train_loss','eval_loss','learning_rate'])
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            with open(self.path, 'a', newline='') as f:
                csv.writer(f).writerow([
                    state.global_step,
                    logs.get('loss',''),
                    logs.get('eval_loss',''),
                    logs.get('learning_rate','')
                ])

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    callbacks=[CSVLoggerCallback('training_log.csv')],
)

print("Starting training!")
trainer.train()
trainer.save_model(OUTPUT_DIR + '/final')

### Step 2: Download your fine-tuned model
Once training completes, the LoRA adapter weights will be saved in the `checkpoints/socratic-sft/final` folder. Download these files to use your custom Socratic tutor!